In [ ]:
import sys
sys.path.insert(0, '/home/mk/CarCV-metrics')

import json
import yaml
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import logging

from utils.model_loader import TrafficCamNetLoader
from utils.data_loader import BDD100KLoader, ImagePreprocessor
from utils.postprocess import decode_detections, apply_nms
from utils.metrics import COCOMetricsComputer

logging.basicConfig(level=logging.INFO)

In [ ]:
with open('configs/experiment/trafficcamnet_eval.yaml') as f:
    config = yaml.safe_load(f)

model_cfg = config['model']
data_cfg = config['data']
print(f"Model: {model_cfg['local_path']}")
print(f"Input size: {model_cfg['input_w']}×{model_cfg['input_h']}")
print(f"Classes: {model_cfg['class_names']}")

In [ ]:
model = TrafficCamNetLoader(model_cfg['local_path'])
print(f"Input shape: {model.input_shape}")
print(f"Output names: {model.output_names}")

In [ ]:
loader = BDD100KLoader(
    ann_json_path=data_cfg['local_ann_json'],
    images_dir=data_cfg['local_images_dir'],
    category_map=data_cfg['category_map'],
    max_images=50  # Small subset for notebook
)

# Display sample image with annotations
img_info = loader.images[0]
img, filename = loader.get_image_by_id(img_info['id'])
boxes = loader.get_annotations_for_image(img_info['id'])

print(f"Image: {filename}, shape: {img.shape}")
print(f"Boxes: {len(boxes)}")

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
for box in boxes:
    x, y, w, h = box['bbox']
    rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='green', linewidth=2)
    plt.gca().add_patch(rect)
plt.title(f"Sample annotation (BDD100K)")
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
preprocessor = ImagePreprocessor(
    input_w=model_cfg['input_w'],
    input_h=model_cfg['input_h'],
    mean_b=model_cfg['mean_b'],
    mean_g=model_cfg['mean_g'],
    mean_r=model_cfg['mean_r'],
    net_scale_factor=model_cfg['net_scale_factor']
)

tensor, scale_x, scale_y = preprocessor.preprocess(img)
outputs = model.infer(tensor)

cov = outputs[model_cfg['output_cov_name']]
bbox_deltas = outputs[model_cfg['output_bbox_name']]

print(f"Coverage shape: {cov.shape}")
print(f"Bbox deltas shape: {bbox_deltas.shape}")
print(f"Coverage range: [{cov.min():.3f}, {cov.max():.3f}]")

In [ ]:
results_json = Path("results/trafficcamnet_eval/results.json")
with open(results_json) as f:
    results = json.load(f)

print("Metrics:")
for key, val in results['metrics'].items():
    print(f"  {key}: {val:.4f}")

print("\nLatency (ms):")
for key, val in results['latency_ms'].items():
    print(f"  {key}: {val:.3f}")

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame([results['metrics']])
print(metrics_df.to_string())

# Check targets
print("\nTarget Metrics Met:")
for key, met in results['target_met'].items():
    print(f"  {key}: {'✓' if met else '✗'}")